# ChakraModel Eval Harness — Kaggle Runner (v4)

Requires:
1. Dataset **ChakraModel_EvalHarness_Code** (v4 — batch/compile OOM fix) attached, e.g. `/kaggle/input/datasets/jayasrikannuchamy/chakramodel-evalharness-code-new`
2. Dataset **ChakraModel_EvalHarness_Weights** (chakra_transformer_best.pth + combo1_best.pth) attached
3. Any polyp datasets you want scored (PolypGen, HyperKvasir, CVC-ClinicDB, etc.) attached — no code changes needed, `dataset_discovery.py` auto-scans `/kaggle/input`
4. Settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On** (timm needs to fetch the ViT-Large pretrained backbone once before the checkpoint overwrites it)

No patch cells in this version — `MAX_BATCH=8` and `torch.compile(mode="default")` are already fixed at the source in the v4 code zip. Do not add cells that edit `run_eval.py`/`eval_engine.py` after staging; if you need to change batch size again, edit `resource_monitor.py` in the source and re-zip.

In [ ]:
!pip install -q timm albumentations opencv-python-headless psutil

In [ ]:
import os, shutil, glob

WORK = '/kaggle/working'

# --- locate the code dataset. Handles: a literal zip upload, OR Kaggle's
#     auto-extracted layout, under ANY dataset slug (e.g. the -new one). ---
code_zip = None
for p in glob.glob('/kaggle/input/**/ChakraModel_EvalHarness_Code*.zip', recursive=True):
    code_zip = p
    break

if code_zip:
    print('Unpacking code from', code_zip)
    shutil.unpack_archive(code_zip, WORK)
else:
    eh_dir = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if os.path.basename(root) == 'eval_harness' and 'run_eval.py' in files:
            eh_dir = root
            break
    assert eh_dir, (
        'Could not find ChakraModel_EvalHarness_Code*.zip or an extracted eval_harness/ '
        'folder anywhere under /kaggle/input — attach the v4 code dataset.'
    )
    dataset_root = os.path.dirname(eh_dir)
    print('Found already-extracted code at', dataset_root, '(Kaggle auto-extracted the zip on upload)')
    shutil.copytree(eh_dir, f'{WORK}/eval_harness', dirs_exist_ok=True)
    src_dir = os.path.join(dataset_root, 'src')
    if os.path.isdir(src_dir):
        shutil.copytree(src_dir, f'{WORK}/src', dirs_exist_ok=True)
    print('Copied eval_harness/ and src/ into', WORK)

# --- sanity check: confirm this IS v4 before wasting GPU hours ---
rm_path = f'{WORK}/eval_harness/resource_monitor.py'
ee_path = f'{WORK}/eval_harness/eval_engine.py'
with open(rm_path) as f:
    rm_src = f.read()
with open(ee_path) as f:
    ee_src = f.read()
assert 'MAX_BATCH         = 8' in rm_src, (
    'Staged resource_monitor.py is NOT v4 (MAX_BATCH != 8). '
    'You attached a stale code dataset version — re-upload ChakraModel_EvalHarness_Code_v4.zip.'
)
assert 'mode="default"' in ee_src, (
    'Staged eval_engine.py is NOT v4 (torch.compile mode != "default"). '
    'You attached a stale code dataset version — re-upload ChakraModel_EvalHarness_Code_v4.zip.'
)
print('Verified: staged code is v4 (MAX_BATCH=8, torch.compile mode=default).')

# --- stage weights where run_eval.py's defaults expect them ---
os.makedirs(f'{WORK}/weights/checkpoints', exist_ok=True)
wanted = {'chakra_transformer_best.pth', 'combo1_best.pth'}
found = set()
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f in wanted and f not in found:
            src = os.path.join(root, f)
            dst = f'{WORK}/weights/checkpoints/{f}'
            shutil.copy(src, dst)
            found.add(f)
            print('Linked', f, 'from', src)
missing = wanted - found
if missing:
    print('WARNING missing weights:', missing, '— attach the ChakraModel_EvalHarness_Weights dataset.')
else:
    print('Both checkpoints staged OK.')

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(' -', torch.cuda.get_device_name(i))
assert torch.cuda.is_available(), 'CUDA GPU not available — select Accelerator = GPU T4 x2 in notebook settings.'

In [ ]:
%cd /kaggle/working
!python eval_harness/run_eval.py

In [2]:
# ============================================================
# UNIVERSAL KAGGLE DATASET FINDER / DIAGNOSER
# ============================================================

import os
from pathlib import Path
from collections import Counter, defaultdict

ROOT = Path("/kaggle/input")


# ------------------------------------------------------------
# SIZE FORMATTER
# ------------------------------------------------------------

def human_size(size):
    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(size)

    for unit in units:
        if size < 1024:
            return f"{size:.2f} {unit}"
        size /= 1024

    return f"{size:.2f} PB"


# ------------------------------------------------------------
# SCAN EVERYTHING
# ------------------------------------------------------------

print("=" * 80)
print("🔎 UNIVERSAL DATASET FINDER")
print("=" * 80)

if not ROOT.exists():
    print(f"❌ Root directory not found: {ROOT}")
    raise SystemExit

files = []
directories = []

total_size = 0
file_extensions = Counter()
directory_file_counts = defaultdict(int)
directory_sizes = defaultdict(int)

max_depth = 0


# ------------------------------------------------------------
# WALK ENTIRE TREE
# ------------------------------------------------------------

for current_root, dirnames, filenames in os.walk(ROOT):

    current_path = Path(current_root)

    directories.append(current_path)

    depth = len(current_path.relative_to(ROOT).parts)
    max_depth = max(max_depth, depth)

    for filename in filenames:

        filepath = current_path / filename

        try:
            size = filepath.stat().st_size
        except (OSError, PermissionError):
            size = 0

        files.append((filepath, size))

        total_size += size

        # Extension
        ext = filepath.suffix.lower()

        if ext:
            file_extensions[ext] += 1
        else:
            file_extensions["[no extension]"] += 1

        # Directory statistics
        directory_file_counts[current_path] += 1
        directory_sizes[current_path] += size


# ------------------------------------------------------------
# BASIC COUNTS
# ------------------------------------------------------------

total_files = len(files)
total_directories = len(directories)

# Root itself is included in directories.
total_subdirectories = max(0, total_directories - 1)

# Branches = directories that contain something below them
branches = [
    d for d in directories
    if any(
        str(f).startswith(str(d) + os.sep)
        for f, _ in files
    )
]

total_branches = len(branches)


# ------------------------------------------------------------
# EMPTY DIRECTORIES
# ------------------------------------------------------------

empty_directories = []

for d in directories:

    try:
        if not any(d.iterdir()):
            empty_directories.append(d)
    except (OSError, PermissionError):
        pass


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\n📊 DATASET SUMMARY")
print("-" * 80)

print(f"📍 ROOT              : {ROOT}")
print(f"📁 Total directories : {total_directories:,}")
print(f"📂 Total subdirs     : {total_subdirectories:,}")
print(f"🌿 Total branches    : {total_branches:,}")
print(f"📄 Total files       : {total_files:,}")
print(f"💾 Total weight      : {human_size(total_size)}")
print(f"📏 Maximum depth     : {max_depth}")
print(f"🕳️ Empty directories : {len(empty_directories):,}")


# ------------------------------------------------------------
# FILE TYPE STATISTICS
# ------------------------------------------------------------

print("\n📦 FILE TYPE COUNTS")
print("-" * 80)

for ext, count in file_extensions.most_common():

    matching_size = sum(
        size for path, size in files
        if (path.suffix.lower() or "[no extension]") == ext
    )

    print(
        f"{ext:<20} "
        f"{count:>10,} files   "
        f"{human_size(matching_size):>12}"
    )


# ------------------------------------------------------------
# DATASET / TOP-LEVEL FOLDER STATISTICS
# ------------------------------------------------------------

print("\n🗂️ TOP-LEVEL DATASETS")
print("-" * 80)

top_level = []

try:
    top_level = sorted(ROOT.iterdir())
except Exception:
    pass

for item in top_level:

    if item.is_dir():

        size = directory_sizes.get(item, 0)
        count = directory_file_counts.get(item, 0)

        # Count all descendants
        descendant_files = 0

        for filepath, _ in files:
            try:
                filepath.relative_to(item)
                descendant_files += 1
            except ValueError:
                pass

        print(
            f"📁 {item.name:<40} "
            f"{descendant_files:>10,} files   "
            f"{human_size(size):>12}"
        )

    else:

        try:
            size = item.stat().st_size
        except:
            size = 0

        print(
            f"📄 {item.name:<40} "
            f"{1:>10,} file     "
            f"{human_size(size):>12}"
        )


# ------------------------------------------------------------
# LARGEST FILES
# ------------------------------------------------------------

print("\n🔥 TOP 20 LARGEST FILES")
print("-" * 80)

largest_files = sorted(
    files,
    key=lambda x: x[1],
    reverse=True
)[:20]

for rank, (filepath, size) in enumerate(largest_files, 1):

    print(
        f"{rank:>2}. "
        f"{human_size(size):>12}  "
        f"{filepath}"
    )


# ------------------------------------------------------------
# LARGEST DIRECTORIES
# ------------------------------------------------------------

print("\n📁 TOP 20 LARGEST DIRECTORIES")
print("-" * 80)

largest_dirs = sorted(
    directory_sizes.items(),
    key=lambda x: x[1],
    reverse=True
)[:20]

for rank, (directory, size) in enumerate(largest_dirs, 1):

    print(
        f"{rank:>2}. "
        f"{human_size(size):>12}  "
        f"{directory}"
    )


# ------------------------------------------------------------
# COMPLETE TREE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("🌳 COMPLETE DIRECTORY TREE")
print("=" * 80)


def print_tree(path, prefix=""):

    try:
        entries = sorted(
            path.iterdir(),
            key=lambda x: (
                not x.is_dir(),
                x.name.lower()
            )
        )
    except (PermissionError, OSError):
        print(prefix + "└── [ACCESS DENIED]")
        return

    for i, entry in enumerate(entries):

        last = i == len(entries) - 1

        branch = "└── " if last else "├── "

        if entry.is_dir():

            count = 0
            size = 0

            for filepath, filesize in files:

                try:
                    filepath.relative_to(entry)
                    count += 1
                    size += filesize
                except ValueError:
                    pass

            print(
                prefix
                + branch
                + f"📁 {entry.name}/ "
                + f"[{count:,} files | {human_size(size)}]"
            )

            extension = "    " if last else "│   "

            print_tree(
                entry,
                prefix + extension
            )

        else:

            try:
                size = entry.stat().st_size
            except:
                size = 0

            print(
                prefix
                + branch
                + f"📄 {entry.name} "
                + f"[{human_size(size)}]"
            )


print(f"\n📁 {ROOT}")
print_tree(ROOT)


# ------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("✅ SCAN COMPLETE")
print("=" * 80)

print(f"""
ROOT              : {ROOT}

DIRECTORIES       : {total_directories:,}
SUBDIRECTORIES    : {total_subdirectories:,}
BRANCHES          : {total_branches:,}
FILES             : {total_files:,}
TOTAL WEIGHT      : {human_size(total_size)}
MAX DEPTH         : {max_depth}
EMPTY DIRECTORIES : {len(empty_directories):,}
FILE TYPES        : {len(file_extensions):,}

Everything under /kaggle/input has been scanned.
""")

🔎 UNIVERSAL DATASET FINDER

📊 DATASET SUMMARY
--------------------------------------------------------------------------------
📍 ROOT              : /kaggle/input
📁 Total directories : 39
📂 Total subdirs     : 38
🌿 Total branches    : 39
📄 Total files       : 3,558
💾 Total weight      : 37.29 GB
📏 Maximum depth     : 5
🕳️ Empty directories : 0

📦 FILE TYPE COUNTS
--------------------------------------------------------------------------------
.jpg                      2,000 files       49.97 MB
.png                      1,120 files       70.34 MB
.avi                        373 files       35.92 GB
.py                          58 files      380.48 KB
.txt                          3 files        2.87 KB
.csv                          2 files       39.87 KB
.pth                          2 files        1.25 GB

🗂️ TOP-LEVEL DATASETS
--------------------------------------------------------------------------------
📁 datasets                                      3,558 files         0.00 B

🔥 

## Sanity peek at the results file
import json, glob
res_files = sorted(glob.glob('/kaggle/working/eval_harness/results/evaluation_results_*.json'))
assert res_files, 'No results file produced. Scroll up in the run cell for the traceback.'
with open(res_files[-1]) as f:
    results = json.load(f)
for r in results:
    print(r.get('dataset'), '->', r.get('vit_large', {}).get('dice'), r.get('error', ''))